In [65]:
import cv2
import mediapipe as mp
import numpy as np

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)

OpenCV: 5.0.0
MediaPipe: 0.10.9
NumPy: 2.4.6


In [66]:
mppose = mp.solutions.pose
mpdraw = mp.solutions.drawing_utils

In [67]:
data = mppose.Pose()

In [68]:
bowling_hand=input("Left or right")

In [69]:
elbow_angles=[]

In [70]:
import math

bowling_hand = bowling_hand.lower()

f = 0
all_frame_data = []
elbow_angles = []
start_found = False
release_found=False
start_frame = None
start_angle = None


video = cv2.VideoCapture("videos/l1.mp4")
while True:
    suc, img = video.read()
    frame_data = []
    if not suc:
        break

    img1 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = data.process(img1)

    if result.pose_landmarks:
        f += 1
        landmarks = result.pose_landmarks.landmark

        if bowling_hand == "r":
            selected_landmarks = [
                mppose.PoseLandmark.RIGHT_SHOULDER,
                mppose.PoseLandmark.RIGHT_ELBOW,
                mppose.PoseLandmark.RIGHT_WRIST
            ]
        else:
            selected_landmarks = [
                mppose.PoseLandmark.LEFT_SHOULDER,
                mppose.PoseLandmark.LEFT_ELBOW,
                mppose.PoseLandmark.LEFT_WRIST
            ]

        for landmark_id in selected_landmarks:
            landmark = landmarks[landmark_id]
            frame_data.append([
                f,
                landmark_id,
                landmark.x,
                landmark.y,
                landmark.z,
                landmark.visibility
            ])

        


        shoulder = frame_data[0]
        elbow = frame_data[1]
        wrist = frame_data[2]


        
        print(shoulder[3],elbow[3])

            


        tolerance = 0.015

        if not start_found:
            if elbow[2]<shoulder[2]:
                if abs(shoulder[3] - elbow[3]) < tolerance:
                    print(shoulder[3],elbow[3])
                    start_frame = f
                    print(frame_data)
                    start_found=True
                

        if start_found:
            sx, sy = shoulder[2], shoulder[3]
            ex, ey = elbow[2], elbow[3]
            wx, wy = wrist[2], wrist[3]

            a = (sx - ex, sy - ey)
            b = (wx - ex, wy - ey)

            dot = a[0] * b[0] + a[1] * b[1]

            mag_a = math.sqrt(a[0]**2 + a[1]**2)
            mag_b = math.sqrt(b[0]**2 + b[1]**2)

            if mag_a != 0 and mag_b != 0:
                value = dot / (mag_a * mag_b)
                value = max(-1, min(1, value))

                angle = math.degrees(
                    math.acos(value)
                )

                if not release_found:
                    elbow_angles.append(angle)

                if f == start_frame:
                    start_angle = angle


            if start_found and not release_found:
                if wrist[3] < shoulder[3] and abs(shoulder[2] - elbow[2]) < 0.02:
                    release_frame = f
                    release_angle = angle
                    release_found = True
            


        mpdraw.draw_landmarks(
            img,
            result.pose_landmarks,
            mppose.POSE_CONNECTIONS
        )

        

    cv2.imshow("img", img)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

video.release()
cv2.destroyAllWindows()

print("TOTAL FRAMES:", f)
print("START FRAME:", start_frame,"RELEASE FRAME:", release_frame)
print("START ANGLE:", start_angle)
print("RELEASE ANGLE:", release_angle)
# print("RECORDED ANGLES:", len(elbow_angles))


extension_angle=int(max(elbow_angles))-int(min(elbow_angles))
print(extension_angle)

0.2346043884754181 0.3602328896522522
0.23217381536960602 0.3399595618247986
0.23013988137245178 0.32253479957580566
0.23024894297122955 0.3280462920665741
0.23012903332710266 0.32820385694503784
0.228547140955925 0.33064356446266174
0.22812899947166443 0.3302311897277832
0.22780463099479675 0.32720425724983215
0.22482195496559143 0.3272720277309418
0.22384648025035858 0.31581082940101624
0.22372114658355713 0.3183861970901489
0.22381193935871124 0.32034382224082947
0.22396618127822876 0.31625983119010925
0.22024376690387726 0.29438328742980957
0.21891950070858002 0.289980411529541
0.2180948406457901 0.29590076208114624
0.21797820925712585 0.2992190718650818
0.2171693742275238 0.2956555485725403
0.21717165410518646 0.29704850912094116
0.21726635098457336 0.2934017777442932
0.21736520528793335 0.3113272488117218
0.2173362672328949 0.31098708510398865
0.21636444330215454 0.31088998913764954
0.21546228229999542 0.30612465739250183
0.21493956446647644 0.30318447947502136
0.2134385854005813

In [71]:
print(elbow_angles)

[153.6141843885558, 157.04572779029425, 151.7906053769152, 153.05401736286473, 150.48706649674452, 152.58915824447863, 152.39152264517884, 147.62466793186084, 147.43011880957783, 149.05841875641175, 149.11569187199512, 155.04571704514876, 155.9330163544192, 155.74708284932566, 157.65555151091513, 158.43210372286782, 164.72426413193514, 167.59551343684137, 168.98770010851078, 173.90749852984854, 175.51980311282222, 174.90179565107965, 175.78168909779944, 179.50546547324987, 179.55110810482824, 179.5107702377733, 177.28854810187738, 177.8514352907418, 169.1237451834872, 169.2924861252372, 170.5153629655232, 166.7362607430085]


In [72]:
print(max(elbow_angles))
print(elbow_angles.index(max(elbow_angles)))

print(min(elbow_angles))
print(elbow_angles.index(min(elbow_angles)))

179.55110810482824
24
147.43011880957783
8


In [73]:
print(int(max(elbow_angles))-int(min(elbow_angles)))

32


In [74]:
for i in elbow_angles:
    if i<167:
        print(i)

153.6141843885558
157.04572779029425
151.7906053769152
153.05401736286473
150.48706649674452
152.58915824447863
152.39152264517884
147.62466793186084
147.43011880957783
149.05841875641175
149.11569187199512
155.04571704514876
155.9330163544192
155.74708284932566
157.65555151091513
158.43210372286782
164.72426413193514
166.7362607430085


In [75]:
print(elbow_angles)

[153.6141843885558, 157.04572779029425, 151.7906053769152, 153.05401736286473, 150.48706649674452, 152.58915824447863, 152.39152264517884, 147.62466793186084, 147.43011880957783, 149.05841875641175, 149.11569187199512, 155.04571704514876, 155.9330163544192, 155.74708284932566, 157.65555151091513, 158.43210372286782, 164.72426413193514, 167.59551343684137, 168.98770010851078, 173.90749852984854, 175.51980311282222, 174.90179565107965, 175.78168909779944, 179.50546547324987, 179.55110810482824, 179.5107702377733, 177.28854810187738, 177.8514352907418, 169.1237451834872, 169.2924861252372, 170.5153629655232, 166.7362607430085]


In [76]:
# Minimum Angle & Index
min_angle = min(elbow_angles)
min_index = elbow_angles.index(min_angle)

# Maximum Angle & Index
max_angle = max(elbow_angles)
max_index = elbow_angles.index(max_angle)

# Output Results
print("MIN ANGLE:", min_angle)
print("MIN INDEX IN LIST:", min_index)
print("---")
print("MAX ANGLE:", max_angle)
print("MAX INDEX IN LIST:", max_index)

MIN ANGLE: 147.43011880957783
MIN INDEX IN LIST: 8
---
MAX ANGLE: 179.55110810482824
MAX INDEX IN LIST: 24


In [77]:
count_88 = elbow_angles.count(88.57166522507079)
count_179 = elbow_angles.count(179.63931063150474)

print("Count of 88.57166522507079:", count_88)
print("Count of 179.63931063150474:", count_179)

Count of 88.57166522507079: 0
Count of 179.63931063150474: 0


In [78]:
for i in elbow_angles:
    print(i)
    

153.6141843885558
157.04572779029425
151.7906053769152
153.05401736286473
150.48706649674452
152.58915824447863
152.39152264517884
147.62466793186084
147.43011880957783
149.05841875641175
149.11569187199512
155.04571704514876
155.9330163544192
155.74708284932566
157.65555151091513
158.43210372286782
164.72426413193514
167.59551343684137
168.98770010851078
173.90749852984854
175.51980311282222
174.90179565107965
175.78168909779944
179.50546547324987
179.55110810482824
179.5107702377733
177.28854810187738
177.8514352907418
169.1237451834872
169.2924861252372
170.5153629655232
166.7362607430085
